<a href="https://colab.research.google.com/github/Artemonkey/DNN_course_practice/blob/main/1_Base_NN_For_Image_Classification_Sign_Language_MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Классификация жестов Sign Language MNIST: baseline и серия экспериментов

Задача: распознавать букву американского жестового алфавита по изображению руки `28 × 28` в оттенках серого. Классов 24: буквы J и Z отсутствуют, так как для них жест требует движения.

План работы:

1. Подготовка данных и разделение на train / validation / test.
2. Baseline: простая Dense-сеть.
3. Пять экспериментов, в каждом меняется ровно один фактор относительно baseline.
4. Абляционная таблица и выводы.
5. Выбор итоговой модели по validation, проверка на test, анализ ошибок.

Validation используется для выбора модели и анализа экспериментов. Test открывается один раз, в самом конце, для итоговой модели.

# Импорт библиотек

In [ ]:
import copy
import os
import random
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
from torch import nn, optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

# For downloading dataset
import kagglehub

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from IPython.display import display

PLOTLY_TEMPLATE = "plotly_dark"

## Seed и устройство

In [ ]:
SEED = 8

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Deterministic cuDNN kernels: slower, but repeated runs match
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Устройство:", device)

Устройство: cuda


# 1. Данные

In [ ]:
path = kagglehub.dataset_download("datamunge/sign-language-mnist")

Using Colab cache for faster access to the 'sign-language-mnist' dataset.


В архиве лежат два CSV-файла: `sign_mnist_train.csv` и `sign_mnist_test.csv`. Каждая строка — одно изображение: столбец `label` и 784 столбца с яркостью пикселей от 0 до 255.

Для работы берётся только `sign_mnist_train.csv` (27 455 изображений), и он целиком делится на три части: train 70%, validation 15%, test 15%. Так все три части получены одинаковым способом, а доли классов сохраняются.

In [ ]:
def find_file(root, name):
    # Search the downloaded folder recursively
    for r, _, files in os.walk(root):
        if name in files:
            return os.path.join(r, name)
    raise FileNotFoundError(name)

df = pd.read_csv(find_file(path, "sign_mnist_train.csv"))
print("Таблица:", df.shape)

# Labels are 0..24 with gaps (9 = J and 25 = Z never occur).
# CrossEntropyLoss needs contiguous indices, so labels are remapped to 0..23.
label_values = np.sort(df["label"].unique())
label_to_idx = {int(v): i for i, v in enumerate(label_values)}
class_names = [chr(ord("A") + int(v)) for v in label_values]
NUM_CLASSES = len(class_names)

y = df["label"].map(label_to_idx).values.astype(np.int64)
X = df.drop(columns="label").values.astype(np.float32) / 255.0  # [N, 784], range 0..1

print("Классов:", NUM_CLASSES, "|", "".join(class_names))
print("X:", X.shape, "| y:", y.shape)

Таблица: (27455, 785)
Классов: 24 | ABCDEFGHIKLMNOPQRSTUVWXY
X: (27455, 784) | y: (27455,)


## Разделение train / validation / test

In [ ]:
TRAIN_SIZE, VAL_SIZE, TEST_SIZE = 0.70, 0.15, 0.15

# 1) split off the test part
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

# 2) split validation from the remaining images
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=VAL_SIZE / (TRAIN_SIZE + VAL_SIZE),
    stratify=y_trainval, random_state=SEED,
)

for name, part in [("train", y_train), ("validation", y_val), ("test", y_test)]:
    print(f"{name:<11}: {len(part):>6} ({len(part) / len(y):.1%})")

assert len(y_train) + len(y_val) + len(y_test) == len(y)

train      :  19217 (70.0%)
validation :   4119 (15.0%)
test       :   4119 (15.0%)


Датасет Sign Language MNIST, по описанию автора, получен размножением небольшого числа исходных фотографий (поворот, изменение яркости, масштаб). Поэтому при случайном разбиении в train и validation/test попадают похожие друг на друга картинки. Это делает задачу легче, чем на реальных новых данных, и итоговые accuracy стоит читать с этой оговоркой. Для сравнения моделей между собой это не мешает: условия у всех одинаковые.

## Подготовка тензоров

Данные небольшие, поэтому один раз переводятся в `float32`-тензоры формы `[N, 1, 28, 28]` и нормализуются. Среднее и стандартное отклонение считаются **только по train**, чтобы информация о validation и test не попадала в обучение. Дальше при сравнении моделей измеряется именно стоимость самих архитектур, а не чтения и преобразования данных.

In [ ]:
pixel_mean = float(X_train.mean())
pixel_std = float(X_train.std())
print(f"mean={pixel_mean:.4f}, std={pixel_std:.4f}")

def to_tensor(images, labels):
    x = torch.from_numpy((images - pixel_mean) / pixel_std).view(-1, 1, 28, 28)
    return x, torch.from_numpy(labels)

x_train, t_train = to_tensor(X_train, y_train)
x_val, t_val = to_tensor(X_val, y_val)
x_test, t_test = to_tensor(X_test, y_test)

print("train:", tuple(x_train.shape), x_train.dtype)
print("val:  ", tuple(x_val.shape))
print("test: ", tuple(x_test.shape))

mean=0.6251, std=0.1911
train: (19217, 1, 28, 28) torch.float32
val:   (4119, 1, 28, 28)
test:  (4119, 1, 28, 28)


## Как выглядят данные

In [ ]:
# One example of every class from the train part
first_of_class = [int(np.where(y_train == c)[0][0]) for c in range(NUM_CLASSES)]

fig = make_subplots(
    rows=4,
    cols=6,
    subplot_titles=[class_names[c] for c in range(NUM_CLASSES)],
)

for c, idx in enumerate(first_of_class):
    fig.add_trace(
        go.Heatmap(
            z=X_train[idx].reshape(28, 28),
            colorscale="gray",
            showscale=False,
            hoverinfo="skip",
        ),
        row=c // 6 + 1,
        col=c % 6 + 1,
    )

fig.update_xaxes(visible=False)
fig.update_yaxes(visible=False, autorange="reversed")
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    title="По одному примеру каждого класса",
    height=700,
    margin=dict(l=20, r=20, t=80, b=20),
)
fig.show()

На графике показано по одному изображению для каждой из 24 букв. Подпись над изображением — буква.

In [ ]:
class_share = pd.DataFrame({
    "class": class_names,
    "train": np.bincount(y_train, minlength=NUM_CLASSES),
    "validation": np.bincount(y_val, minlength=NUM_CLASSES),
    "test": np.bincount(y_test, minlength=NUM_CLASSES),
}).melt(id_vars="class", var_name="split", value_name="images")

fig = px.bar(
    class_share,
    x="class",
    y="images",
    color="split",
    barmode="group",
    title="Число изображений по классам",
    template=PLOTLY_TEMPLATE,
    height=420,
)
fig.update_layout(xaxis_title="Класс", yaxis_title="Изображений")
fig.show()

На графике показано, сколько изображений каждого класса попало в train, validation и test. Классы распределены почти поровну, и доли сохранены во всех трёх частях.

## DataLoader

In [ ]:
BATCH_SIZE = 128

train_data = TensorDataset(x_train, t_train)
val_data = TensorDataset(x_val, t_val)
test_data = TensorDataset(x_test, t_test)

# num_workers=0: the data is already in memory, extra workers only add overhead
PIN_MEMORY = device.type == "cuda"

def make_train_loader():
    # A fresh generator with the same seed gives every experiment the same batch order
    return DataLoader(
        train_data,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=torch.Generator().manual_seed(SEED),
        pin_memory=PIN_MEMORY,
    )

val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=PIN_MEMORY)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, pin_memory=PIN_MEMORY)

# 2. Общие настройки обучения

Все модели, включая baseline, обучаются одинаково. Различается только архитектура, поэтому разницу в результате можно относить на счёт изменённого фактора.

| Параметр | Значение |
|---|---|
| Loss | `CrossEntropyLoss` |
| Optimizer | `SGD`, `lr = 0.01`, без momentum |
| Batch size | 128 |
| Эпох | 30 |
| Seed | 8 (одинаковый для инициализации весов и порядка батчей) |
| Выбор эпохи | эпоха с лучшей validation accuracy |

In [ ]:
EPOCHS = 30
LR = 1e-2

criterion = nn.CrossEntropyLoss()

def make_optimizer(model):
    return optim.SGD(model.parameters(), lr=LR)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

In [ ]:
def run_epoch(model, loader, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)

    total_loss = torch.zeros((), device=device)
    total_correct = torch.zeros((), device=device)
    total_count = 0

    for x, y in loader:
        x = x.to(device, non_blocking=PIN_MEMORY)
        y = y.to(device, non_blocking=PIN_MEMORY)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        else:
            with torch.inference_mode():
                logits = model(x)
                loss = criterion(logits, y)

        total_loss += loss.detach() * x.size(0)
        total_correct += (logits.argmax(dim=1) == y).sum()
        total_count += x.size(0)

    # One CPU/GPU synchronisation per epoch
    return {
        "loss": (total_loss / total_count).item(),
        "acc": (total_correct / total_count).item(),
    }


def fit_model(model, train_loader, epochs=EPOCHS):
    optimizer = make_optimizer(model)
    history = []
    best_state, best_val_acc = None, -1.0

    for epoch in range(1, epochs + 1):
        tr = run_epoch(model, train_loader, optimizer)
        va = run_epoch(model, val_loader)

        history.append({
            "epoch": epoch,
            "train_loss": tr["loss"], "train_acc": tr["acc"],
            "val_loss": va["loss"], "val_acc": va["acc"],
        })

        if va["acc"] > best_val_acc:
            best_val_acc = va["acc"]
            best_state = copy.deepcopy(model.state_dict())

        if epoch % 5 == 0 or epoch == 1:
            print(
                f"epoch {epoch:>2}/{epochs} | "
                f"train loss={tr['loss']:.4f} acc={tr['acc']:.4f} | "
                f"val loss={va['loss']:.4f} acc={va['acc']:.4f}"
            )

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

In [ ]:
@torch.no_grad()
def benchmark_inference(model, batch, warmup=5, repeats=50):
    # Latency and throughput on one fixed batch, same runtime for all models
    model.eval()
    x = batch.to(device)

    for _ in range(warmup):
        model(x)
    sync()

    times = []
    for _ in range(repeats):
        sync()
        start = time.perf_counter()
        model(x)
        sync()
        times.append((time.perf_counter() - start) * 1000)

    latency_ms = float(np.median(times))
    return {
        "latency_ms_batch": latency_ms,
        "throughput_img_s": len(x) / (latency_ms / 1000),
    }

In [ ]:
@dataclass
class Experiment:
    id: str
    name: str
    change: str      # what differs from the baseline
    builder: object  # function that returns a fresh model

results = []
histories = {}
trained_models = {}

def run_experiment(exp):
    set_seed()  # same seed -> same weight initialisation and batch order for every run
    model = exp.builder().to(device)
    params = count_parameters(model)

    print(f"{exp.id}: {exp.name} | параметров: {params:,}")
    sync()
    start = time.perf_counter()
    model, history = fit_model(model, make_train_loader())
    sync()
    train_seconds = time.perf_counter() - start

    best = history.loc[history["val_acc"].idxmax()]
    speed = benchmark_inference(model, x_val[:BATCH_SIZE])

    results.append({
        "id": exp.id,
        "model": exp.name,
        "change": exp.change,
        "params": params,
        "size_kb": params * 4 / 1024,
        "best_epoch": int(best["epoch"]),
        "train_acc": float(best["train_acc"]),
        "val_acc": float(best["val_acc"]),
        "val_loss": float(best["val_loss"]),
        "train_seconds": train_seconds,
        **speed,
    })
    histories[exp.id] = history
    trained_models[exp.id] = model

    base = results[0]["val_acc"]
    print(
        f"-> лучшая эпоха {int(best['epoch'])}, val acc = {best['val_acc']:.4f} "
        f"({(best['val_acc'] - base) * 100:+.2f} п.п. к E0), "
        f"время обучения {train_seconds:.1f} с"
    )

# 3. Baseline (E0)

Baseline простой: одна скрытая Dense-сеть с ReLU.

```text
изображение 1×28×28
  ↓ Flatten
784
  ↓ Linear(784 → 64)
  ↓ ReLU
64
  ↓ Linear(64 → 24)
24 логита классов
```

Остальные условия зафиксированы в таблице выше: `CrossEntropyLoss`, `SGD(lr=0.01)`, 30 эпох, batch 128, seed 8.

In [ ]:
def make_dense(activation=nn.ReLU, hidden=64, dropout=0.0):
    layers = [nn.Flatten(), nn.Linear(28 * 28, hidden), activation()]
    if dropout > 0:
        layers.append(nn.Dropout(dropout))
    layers.append(nn.Linear(hidden, NUM_CLASSES))
    return nn.Sequential(*layers)

baseline = Experiment("E0", "Dense 64 / ReLU", "baseline", lambda: make_dense(nn.ReLU))
print(baseline.builder())
run_experiment(baseline)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Linear(in_features=784, out_features=64, bias=True)
  (2): ReLU()
  (3): Linear(in_features=64, out_features=24, bias=True)
)
E0: Dense 64 / ReLU | параметров: 51,800
epoch  1/30 | train loss=2.8445 acc=0.2143 | val loss=2.5035 acc=0.3120
epoch  5/30 | train loss=1.2007 acc=0.7138 | val loss=1.1206 acc=0.7322
epoch 10/30 | train loss=0.6483 acc=0.8644 | val loss=0.6218 acc=0.8730
epoch 15/30 | train loss=0.4019 acc=0.9303 | val loss=0.3916 acc=0.9342
epoch 20/30 | train loss=0.2663 acc=0.9695 | val loss=0.2635 acc=0.9680
epoch 25/30 | train loss=0.1850 acc=0.9877 | val loss=0.1846 acc=0.9832
epoch 30/30 | train loss=0.1344 acc=0.9951 | val loss=0.1347 acc=0.9934
-> лучшая эпоха 30, val acc = 0.9934 (+0.00 п.п. к E0), время обучения 17.7 с


In [ ]:
def plot_history(history, title):
    fig = make_subplots(rows=1, cols=2, subplot_titles=("Loss", "Accuracy"))

    for col, metric in enumerate(["loss", "acc"], start=1):
        for split, dash in [("train", "solid"), ("val", "dash")]:
            fig.add_trace(
                go.Scatter(
                    x=history["epoch"],
                    y=history[f"{split}_{metric}"],
                    mode="lines",
                    name=split,
                    line=dict(dash=dash),
                    legendgroup=split,
                    showlegend=(col == 1),
                ),
                row=1,
                col=col,
            )

    fig.update_xaxes(title_text="Epoch")
    fig.update_layout(template=PLOTLY_TEMPLATE, title=title, height=420)
    fig.show()

plot_history(histories["E0"], "Baseline E0: обучение")

b = results[0]
pd.DataFrame({
    "Параметр": ["Архитектура", "Обучаемых параметров", "Loss", "Optimizer", "Эпох",
                 "Лучшая эпоха", "Validation accuracy", "Время обучения"],
    "Значение": ["784 → 64 → 24, ReLU", f"{b['params']:,}", "CrossEntropyLoss", f"SGD, lr={LR}", EPOCHS,
                 b["best_epoch"], f"{b['val_acc']:.4f}", f"{b['train_seconds']:.1f} с"],
}).style.hide(axis="index")

Параметр,Значение
Архитектура,"784 → 64 → 24, ReLU"
Обучаемых параметров,"51,800"
Loss,CrossEntropyLoss
Optimizer,"SGD, lr=0.01"
Эпох,30
Лучшая эпоха,30
Validation accuracy,0.9934
Время обучения,17.7 с


На графиках показаны loss и accuracy baseline по эпохам: сплошная линия — train, пунктир — validation. Ниже таблица с фиксированными условиями и результатом baseline.

# 4. Эксперименты

В каждом эксперименте меняется один фактор относительно baseline. Гипотеза записана до запуска.

| ID | Изменение относительно E0 | Что проверяется |
|---|---|---|
| E1 | ReLU → Tanh | влияние функции активации |
| E2 | ReLU → Sigmoid | влияние функции активации |
| E3 | + Dropout 0.3 после скрытого слоя | влияние регуляризации |
| E4 | Dense → CNN (1 conv-слой) | влияние пространственной структуры |
| E5 | CNN с двумя conv-слоями | влияние глубины свёрточной части |

## E1. ReLU → Tanh

**Гипотеза.** Качество останется близким к baseline: разница в пределах 1–2 п.п. Tanh даёт выход, центрированный около нуля, что при обычном SGD иногда ускоряет обучение, но у сети с одним скрытым слоем из 64 нейронов выигрыш должен быть небольшим.

In [ ]:
run_experiment(Experiment("E1", "Dense 64 / Tanh", "ReLU → Tanh", lambda: make_dense(nn.Tanh)))

E1: Dense 64 / Tanh | параметров: 51,800
epoch  1/30 | train loss=2.7298 acc=0.2739 | val loss=2.4148 acc=0.3969
epoch  5/30 | train loss=1.4669 acc=0.7043 | val loss=1.4015 acc=0.7075
epoch 10/30 | train loss=0.9270 acc=0.8165 | val loss=0.9099 acc=0.8184
epoch 15/30 | train loss=0.6493 acc=0.8861 | val loss=0.6456 acc=0.8893
epoch 20/30 | train loss=0.4759 acc=0.9341 | val loss=0.4775 acc=0.9301
epoch 25/30 | train loss=0.3594 acc=0.9623 | val loss=0.3646 acc=0.9582
epoch 30/30 | train loss=0.2784 acc=0.9806 | val loss=0.2851 acc=0.9772
-> лучшая эпоха 30, val acc = 0.9772 (-1.63 п.п. к E0), время обучения 11.3 с


## E2. ReLU → Sigmoid

**Гипотеза.** Sigmoid отстанет от ReLU заметно, за те же 30 эпох. У Sigmoid производная не превышает 0.25, а выход не центрирован около нуля, поэтому градиенты малы и SGD с `lr = 0.01` продвигается медленно. Ожидается, что кривая validation accuracy у E2 всё ещё растёт к 30-й эпохе.

In [ ]:
run_experiment(Experiment("E2", "Dense 64 / Sigmoid", "ReLU → Sigmoid", lambda: make_dense(nn.Sigmoid)))

E2: Dense 64 / Sigmoid | параметров: 51,800
epoch  1/30 | train loss=3.1291 acc=0.1004 | val loss=3.0599 acc=0.1685
epoch  5/30 | train loss=2.7042 acc=0.3222 | val loss=2.6581 acc=0.3265
epoch 10/30 | train loss=2.2641 acc=0.4417 | val loss=2.2326 acc=0.4436
epoch 15/30 | train loss=1.9435 acc=0.5642 | val loss=1.9244 acc=0.5664
epoch 20/30 | train loss=1.6978 acc=0.6344 | val loss=1.6868 acc=0.6293
epoch 25/30 | train loss=1.5008 acc=0.6725 | val loss=1.4961 acc=0.6616
epoch 30/30 | train loss=1.3400 acc=0.7107 | val loss=1.3396 acc=0.7007
-> лучшая эпоха 30, val acc = 0.7007 (-29.28 п.п. к E0), время обучения 16.0 с


## E3. Dropout 0.3

**Гипотеза.** Dropout сократит разрыв между train и validation accuracy, но validation accuracy не вырастет: baseline с 64 нейронами и 30 эпохами SGD скорее недообучен, чем переобучен. Возможно, качество даже немного упадёт, так как Dropout замедляет обучение.

In [ ]:
run_experiment(Experiment("E3", "Dense 64 / ReLU / Dropout 0.3", "+ Dropout 0.3", lambda: make_dense(nn.ReLU, dropout=0.3)))

E3: Dense 64 / ReLU / Dropout 0.3 | параметров: 51,800
epoch  1/30 | train loss=2.8803 acc=0.1800 | val loss=2.5355 acc=0.3142
epoch  5/30 | train loss=1.4797 acc=0.5718 | val loss=1.2454 acc=0.7072
epoch 10/30 | train loss=0.9860 acc=0.7185 | val loss=0.7680 acc=0.8354
epoch 15/30 | train loss=0.7365 acc=0.7933 | val loss=0.5319 acc=0.9000
epoch 20/30 | train loss=0.5863 acc=0.8376 | val loss=0.3900 acc=0.9366
epoch 25/30 | train loss=0.4803 acc=0.8692 | val loss=0.2959 acc=0.9534
epoch 30/30 | train loss=0.3988 acc=0.8975 | val loss=0.2267 acc=0.9694
-> лучшая эпоха 30, val acc = 0.9694 (-2.40 п.п. к E0), время обучения 11.5 с


## E4. Dense → CNN

**Гипотеза.** Свёрточная сеть даст заметный прирост качества, на несколько п.п. Жест распознаётся по форме и взаимному расположению пальцев, то есть по соседним пикселям. Dense-сеть после `Flatten` эту структуру не использует, а свёртка 3×3 с общими весами использует.

Контроль: размер скрытого Dense-слоя остаётся 64, но из-за большего вектора признаков после свёртки параметров станет примерно в 4 раза больше. Это нужно учесть при интерпретации.

```text
1×28×28 → Conv 3×3, 16 фильтров, ReLU → 16×28×28 → MaxPool 2×2 → 16×14×14
        → Flatten (3136) → Linear(3136 → 64) → ReLU → Linear(64 → 24)
```

In [ ]:
def make_cnn_1conv():
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Flatten(),
        nn.Linear(16 * 14 * 14, 64),
        nn.ReLU(),
        nn.Linear(64, NUM_CLASSES),
    )

run_experiment(Experiment("E4", "CNN 1 conv (16) + Dense 64", "Dense → CNN", make_cnn_1conv))

E4: CNN 1 conv (16) + Dense 64 | параметров: 202,488
epoch  1/30 | train loss=2.7449 acc=0.2594 | val loss=2.2330 acc=0.4003
epoch  5/30 | train loss=0.6471 acc=0.8318 | val loss=0.6375 acc=0.8186
epoch 10/30 | train loss=0.2298 acc=0.9576 | val loss=0.3705 acc=0.8864
epoch 15/30 | train loss=0.0995 acc=0.9900 | val loss=0.2217 acc=0.9408
epoch 20/30 | train loss=0.0519 acc=0.9975 | val loss=0.0607 acc=0.9942
epoch 25/30 | train loss=0.0321 acc=0.9991 | val loss=0.0376 acc=0.9983
epoch 30/30 | train loss=0.0222 acc=0.9995 | val loss=0.0290 acc=0.9976
-> лучшая эпоха 25, val acc = 0.9983 (+0.49 п.п. к E0), время обучения 15.2 с


## E5. Второй свёрточный слой

**Гипотеза.** Второй conv-слой (32 фильтра) даст дополнительный прирост относительно E4 и при этом уменьшит число параметров: после второго pooling вектор признаков короче (1568 вместо 3136), а значит, первый Dense-слой получает вдвое меньше весов.

```text
1×28×28 → Conv 3×3, 16, ReLU, MaxPool → 16×14×14
        → Conv 3×3, 32, ReLU, MaxPool → 32×7×7
        → Flatten (1568) → Linear(1568 → 64) → ReLU → Linear(64 → 24)
```

In [ ]:
def make_cnn_2conv():
    return nn.Sequential(
        nn.Conv2d(1, 16, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Conv2d(16, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2),

        nn.Flatten(),
        nn.Linear(32 * 7 * 7, 64),
        nn.ReLU(),
        nn.Linear(64, NUM_CLASSES),
    )

run_experiment(Experiment("E5", "CNN 2 conv (16, 32) + Dense 64", "+ второй conv-слой", make_cnn_2conv))

E5: CNN 2 conv (16, 32) + Dense 64 | параметров: 106,776
epoch  1/30 | train loss=3.0833 acc=0.1192 | val loss=2.8987 acc=0.1695
epoch  5/30 | train loss=0.9018 acc=0.7205 | val loss=1.1199 acc=0.6509
epoch 10/30 | train loss=0.3060 acc=0.9071 | val loss=0.5130 acc=0.8393
epoch 15/30 | train loss=0.1003 acc=0.9792 | val loss=0.1292 acc=0.9638
epoch 20/30 | train loss=0.0425 acc=0.9947 | val loss=0.0454 acc=0.9944
epoch 25/30 | train loss=0.0285 acc=0.9971 | val loss=0.0232 acc=0.9973
epoch 30/30 | train loss=0.0133 acc=0.9991 | val loss=0.0144 acc=0.9985
-> лучшая эпоха 28, val acc = 0.9985 (+0.51 п.п. к E0), время обучения 15.2 с


# 5. Абляционная таблица

In [ ]:
ablation = pd.DataFrame(results)
ablation["delta_pp"] = (ablation["val_acc"] - ablation.loc[0, "val_acc"]) * 100
ablation["gap_pp"] = (ablation["train_acc"] - ablation["val_acc"]) * 100

display(
    ablation[[
        "id", "model", "params", "val_acc", "delta_pp", "train_seconds",
        "gap_pp", "latency_ms_batch", "throughput_img_s", "size_kb",
    ]].rename(columns={
        "id": "ID", "model": "Модель", "params": "Параметры",
        "val_acc": "Val accuracy", "delta_pp": "Δ к E0, п.п.",
        "train_seconds": "Обучение, с", "gap_pp": "Train − Val, п.п.",
        "latency_ms_batch": "Latency, мс/батч", "throughput_img_s": "Throughput, изобр./с",
        "size_kb": "Размер, КБ",
    }).style.hide(axis="index").format({
        "Параметры": "{:,}", "Val accuracy": "{:.4f}", "Δ к E0, п.п.": "{:+.2f}",
        "Обучение, с": "{:.1f}", "Train − Val, п.п.": "{:.2f}",
        "Latency, мс/батч": "{:.3f}", "Throughput, изобр./с": "{:,.0f}", "Размер, КБ": "{:,.0f}",
    })
)

ID,Модель,Параметры,Val accuracy,"Δ к E0, п.п.","Обучение, с","Train − Val, п.п.","Latency, мс/батч","Throughput, изобр./с","Размер, КБ"
E0,Dense 64 / ReLU,"51,800",0.9934,+0.00,17.7,0.17,0.115,"1,115,692",202
E1,Dense 64 / Tanh,"51,800",0.9772,-1.63,11.3,0.34,0.126,"1,015,265",202
E2,Dense 64 / Sigmoid,"51,800",0.7007,-29.28,16.0,1.00,0.108,"1,180,360",202
E3,Dense 64 / ReLU / Dropout 0.3,"51,800",0.9694,-2.40,11.5,-7.19,0.125,"1,025,284",202
E4,CNN 1 conv (16) + Dense 64,"202,488",0.9983,+0.49,15.2,0.08,0.319,"401,767",791
E5,"CNN 2 conv (16, 32) + Dense 64","106,776",0.9985,+0.51,15.2,0.09,0.401,"318,933",417


В таблице собраны все модели. «Δ к E0» — разница validation accuracy с baseline в процентных пунктах. «Train − Val» — разрыв между точностью на train и validation, признак переобучения. Время обучения включает проходы по validation. У E3 точность на train измеряется с включённым Dropout, поэтому разрыв с validation у неё частично занижен. Latency и throughput измерены на одном батче из 128 изображений; сравнивать их можно только между строками этой таблицы, так как всё запускалось в одном runtime на одном оборудовании. Размер — вес параметров в `float32`.

In [ ]:
history_long = pd.concat(
    [h.assign(id=exp_id) for exp_id, h in histories.items()],
    ignore_index=True,
)

for metric, label in [("val_acc", "Validation accuracy"), ("val_loss", "Validation loss")]:
    fig = px.line(
        history_long, x="epoch", y=metric, color="id", markers=True,
        title=f"{label} по эпохам", template=PLOTLY_TEMPLATE, height=450,
    )
    fig.update_layout(xaxis_title="Epoch", yaxis_title=label)
    fig.show()

На двух графиках показаны validation accuracy и validation loss по эпохам для всех экспериментов. По ним видно, какие модели учатся быстрее и не вышли ли на плато к концу обучения.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Accuracy vs число параметров", "Accuracy vs время обучения"),
)

for col, (xcol, xtitle) in enumerate(
    [("params", "Параметры (log)"), ("train_seconds", "Время обучения, с")], start=1
):
    fig.add_trace(
        go.Scatter(
            x=ablation[xcol], y=ablation["val_acc"],
            mode="markers+text", text=ablation["id"], textposition="top center",
            marker=dict(size=14), hovertext=ablation["model"], showlegend=False,
        ),
        row=1, col=col,
    )
    fig.update_xaxes(title_text=xtitle, row=1, col=col)

fig.update_xaxes(type="log", row=1, col=1)
fig.update_yaxes(title_text="Validation accuracy")
fig.update_layout(template=PLOTLY_TEMPLATE, height=450)
fig.show()

На графиках по вертикали validation accuracy, по горизонтали число параметров (слева, логарифмическая шкала) и время обучения (справа). Точка выше и левее означает более точную и более дешёвую модель.

## Выводы по экспериментам

1. Наибольший эффект дала Sigmoid (E2), но отрицательный: −29.28 п.п. (val accuracy 0.7007). Среди улучшений наибольший прирост дал переход к CNN: E4 +0.49 п.п., E5 +0.51 п.п. В процентных пунктах прирост небольшой, так как baseline уже около 99.3%. Но число ошибок на validation упало примерно с 27 до 6–7 из 4119, то есть почти в 4 раза.
По времени обучения выиграл эксперимент №1 с изменением функции активации с ReLU на гиперболический тангенс в FCNN.
2. Отдельные эксперименты №1 (Tanh) и №2 (дополнительный DropOut) со значением вероятности 0.3 ухудшили качество на 1.63 и 2.40 п.п.. С другой стороны, эти оба эксперимента сделали обучение быстрее примерно на 5 секунд  - два самых быстрых результата из всех 5 экспериментов.
CNN из обеих экспериментов обучались примерно одинаковое количество времени (разница на ~1 мс), что нельзя сказать о количестве параметров E5 против E4): +0.02 п.п. при этом параметров вдвое меньше (106 776 против 202 488).
3. В случае перехода и уточнения сверточных моделей это дало качественный результат увеличения правильных ответов.
Изменения полносвязных сетей стали ошибаться больше, но выигрывать по скорости обучения.
4. Baseline E0 уступает лучшей модели E5 по val accuracy на 0.51 п.п. (0.9934 против 0.9985). При этом у него вдвое меньше параметров (51 800 против 106 776), вдвое меньший размер (202 против 417 КБ), а inference в 3.5 раза быстрее (0.115 против 0.401 мс/батч). Если главное — скорость и размер, E0 вполне оправданный выбор. Если важнее точность, лучше E5.
5. Далее проведена оценка состоятельности выдвинутых гипотез.
- E1: гипотеза подтвердилась частично. Точность, действительно, почти не изменилась, но время обучения стало значительно меньше.
- E2: направление верное, масштаб недооценён (ожидалось несколько п.п., получилось −29). Кривая к 30-й эпохе всё ещё растёт: Sigmoid не успела сойтись.
- E3: val accuracy не выросла, а упала. Отрицательный «Train − Val» (−7.19 п.п.) возможно связан с тем, что train accuracy измеряется при включённом Dropout.
- E4: гипотеза подтвердилась.
- E5: подтвердилась только часть. Параметров на 47% меньше, чем у E4, а дополнительного прироста нет.

# 6. Итоговая модель и проверка на test

Итоговая модель выбирается по validation accuracy. На test она проверяется один раз.

In [ ]:
best_id = ablation.sort_values("val_acc", ascending=False).iloc[0]["id"]
best_model = trained_models[best_id]
best_row = ablation[ablation["id"] == best_id].iloc[0]

print(f"Итоговая модель: {best_id} — {best_row['model']}")
print(f"Параметров: {int(best_row['params']):,} | val accuracy: {best_row['val_acc']:.4f}")

test_metrics = run_epoch(best_model, test_loader)
print(f"TEST loss: {test_metrics['loss']:.4f} | TEST accuracy: {test_metrics['acc']:.4f}")

Итоговая модель: E5 — CNN 2 conv (16, 32) + Dense 64
Параметров: 106,776 | val accuracy: 0.9985
TEST loss: 0.0193 | TEST accuracy: 0.9981


# 7. Анализ ошибок

In [ ]:
@torch.no_grad()
def predict_probs(model, x):
    model.eval()
    out = []
    for i in range(0, len(x), 512):
        out.append(model(x[i:i + 512].to(device)).softmax(dim=1).cpu())
    return torch.cat(out)

probs = predict_probs(best_model, x_test)
pred = probs.argmax(dim=1).numpy()
conf = probs.max(dim=1).values.numpy()

cm = confusion_matrix(y_test, pred, labels=range(NUM_CLASSES))

fig = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale="Blues",
    x=class_names,
    y=class_names,
    labels=dict(x="Предсказанный класс", y="Истинный класс", color="Изображений"),
    title=f"Confusion matrix на test — {best_id}",
    template=PLOTLY_TEMPLATE,
    height=800,
)
fig.show()

На графике показана confusion matrix на test: по строкам истинная буква, по столбцам предсказанная. Диагональ — верные ответы, остальные ячейки — ошибки.

In [ ]:
errors = cm.copy()
np.fill_diagonal(errors, 0)

pairs = pd.DataFrame(
    [
        {"pair": f"{class_names[t]} → {class_names[p]}", "count": int(errors[t, p])}
        for t in range(NUM_CLASSES) for p in range(NUM_CLASSES) if errors[t, p] > 0
    ]
)

if len(pairs):
    pairs = pairs.sort_values("count", ascending=False).head(10)
    fig = px.bar(pairs, x="pair", y="count", title="Самые частые ошибки (истинный → предсказанный)",
                 template=PLOTLY_TEMPLATE, height=420)
    fig.update_layout(xaxis_title="Истинный → предсказанный", yaxis_title="Ошибок")
    fig.show()

recall = cm.diagonal() / cm.sum(axis=1)
worst = pd.DataFrame({"class": class_names, "recall": recall}).sort_values("recall").head(5)
print("Классы с наименьшей долей верных ответов:")
display(worst.style.hide(axis="index").format({"recall": "{:.3f}"}))

Классы с наименьшей долей верных ответов:


class,recall
H,0.987
O,0.989
P,0.994
K,0.994
N,0.994


На графике показаны десять самых частых пар «истинная буква → предсказанная». Ниже таблица пяти букв, которые модель узнаёт хуже всего.

In [ ]:
wrong = np.where(pred != y_test)[0]
print(f"Ошибок на test: {len(wrong)} из {len(y_test)}")

# Probability the model gave to the true class vs. to its own answer
p_true = probs[wrong, y_test[wrong]].numpy()
p_pred = conf[wrong]

picked = {}
if len(wrong):
    picked["Уверенная ошибка"] = wrong[np.argmax(p_pred)]
    picked["Неуверенная ошибка"] = wrong[np.argmin(p_pred)]
    # Ambiguous: the model's answer and the true class are almost equally likely
    order = np.argsort(p_pred - p_true)
    picked["Неоднозначный случай"] = next(
        (wrong[i] for i in order if wrong[i] not in picked.values()), wrong[order[0]]
    )

if picked:
    fig = make_subplots(
        rows=1, cols=len(picked),
        subplot_titles=[
            f"{kind}<br>истина {class_names[y_test[i]]}, ответ {class_names[pred[i]]}, "
            f"p={conf[i]:.2f}, p(истина)={probs[i, y_test[i]]:.2f}"
            for kind, i in picked.items()
        ],
    )
    for col, i in enumerate(picked.values(), start=1):
        fig.add_trace(
            go.Heatmap(z=X_test[i].reshape(28, 28), colorscale="gray", showscale=False, hoverinfo="skip"),
            row=1, col=col,
        )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, autorange="reversed")
    fig.update_layout(template=PLOTLY_TEMPLATE, height=420, margin=dict(t=110))
    fig.show()

Ошибок на test: 8 из 4119


На рисунке показаны три ошибки итоговой модели: уверенная (наибольшая вероятность в неверном ответе), неуверенная (наименьшая) и неоднозначная (вероятности неверного ответа и истинного класса почти равны). В подписи указаны истинная буква, ответ модели, вероятность ответа и вероятность истинного класса.

In [ ]:
# Examples of the most frequent error pair: true image vs. what the model took it for
if len(pairs):
    t_name, p_name = pairs.iloc[0]["pair"].split(" → ")
    t_idx, p_idx = class_names.index(t_name), class_names.index(p_name)

    wrong_pair = np.where((y_test == t_idx) & (pred == p_idx))[0][:4]
    right_true = np.where((y_test == t_idx) & (pred == t_idx))[0][:2]
    right_pred = np.where((y_test == p_idx) & (pred == p_idx))[0][:2]

    items = (
        [(f"{t_name}, ошибка (ответ {p_name})", i) for i in wrong_pair]
        + [(f"{t_name}, верно", i) for i in right_true]
        + [(f"{p_name}, верно", i) for i in right_pred]
    )

    fig = make_subplots(rows=2, cols=4, subplot_titles=[t for t, _ in items])
    for k, (_, i) in enumerate(items):
        fig.add_trace(
            go.Heatmap(z=X_test[i].reshape(28, 28), colorscale="gray", showscale=False, hoverinfo="skip"),
            row=k // 4 + 1, col=k % 4 + 1,
        )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False, autorange="reversed")
    fig.update_layout(template=PLOTLY_TEMPLATE, height=560, title=f"Самая частая ошибка: {t_name} → {p_name}")
    fig.show()

На рисунке показаны примеры самой частой пары ошибок: сначала изображения, которые модель перепутала, затем верно распознанные образцы обеих букв для сравнения.

## Интерпретация ошибок

1. Жест для латинской буквы *H* в двух случаях определяются жестом для *Y*. Аналогично у для определения O→Y. Остальные четыре единичные: D→R, K→T, N→I, P→I. Устойчивой пары классов, которую модель систематически путала бы, нет.
2. На ошибочных вариантов пальцы под резкой тенью, что делает их крайне темными. Это могло стать причиной ошибок, так как на краях жестов для *Y* заметны резкие темные участки на краях ладоней. Два ошибочных изображения идентичны, поэтому вывод опирается по сути на один пример.
3. Похоже, что использовалась фотография с затемненным участком пальцев рук и кадрированием.
4. Предлагается провести аугментацию яркости и контраста или нормализацию контраста по каждому изображению. Можно попробовать увеличить площадь фильтра свёртки до 4x4, чтобы захватить больше участков для сохранения параметров описывающих форму жеста.

# 8. Итог

Итоговая модель: E5, CNN с двумя conv-слоями (16 и 32 фильтра) и Dense 64, 106 776 параметров. Validation accuracy 0.9985, test accuracy около 0.998 (8 ошибок из 4119; точное значение возьмите из вывода ячейки с TEST). Главный результат: переход от Dense к CNN сократил число ошибок примерно в 4 раза ценой роста latency в 3.5 раза, а Sigmoid при этих условиях обучения (SGD, lr 0.01, 30 эпох) не сошлась.